## 🎯 Learning Objectives
* Design and implement a production-ready CrewAI system integrating advanced features.
* Effectively utilize agent memory to maintain context and improve iterative task execution.
* Develop and apply custom evaluation functions to ensure high-quality task outputs.
* Orchestrate complex multi-agent workflows using CrewAI's Flow capabilities.
* Structure and comment CrewAI code for clarity, maintainability, and production deployment.


## Exercise: Production-Ready Crew with Memory, Evals, and Flows

### Task Description

Your goal is to build a robust, production-ready CrewAI system designed to perform comprehensive market research for a new product launch. Imagine a tech startup is launching an innovative AI-powered personal assistant, and they need a detailed market analysis to inform their strategy.

Your Crew will consist of specialized agents working collaboratively to achieve the following:

1.  **Market Overview**: Analyze the current landscape of AI personal assistants, identifying key players, their core features, pricing models, and target demographics.
2.  **Competitive Deep Dive**: Select the top 3 most relevant competitors identified in the overview and perform a detailed analysis of their strengths, weaknesses, and unique selling propositions.
3.  **Strategic Recommendations**: Synthesize all gathered information into a concise market research report, including actionable recommendations for the new AI assistant's positioning, feature set, and potential market entry strategy.

### Requirements

Your solution must incorporate the following advanced CrewAI features:

*   **Agents with Memory**: All agents involved in the research process must have `memory=True` enabled to ensure they can recall previous interactions and findings, allowing for iterative refinement and context retention across tasks.
*   **Custom Evaluation Functions**: Each major task (Market Overview, Competitive Deep Dive, Strategic Recommendations) must have a custom `evaluation_function` defined. This function should assess the quality, completeness, and relevance of the task's output. For this exercise, a simple keyword-based or structure-based evaluation is acceptable.
*   **Flows for Orchestration**: Utilize CrewAI's `Flow` mechanism to define the sequence and dependencies of your tasks. The flow should ensure that the competitive deep dive only begins after the market overview is complete, and the recommendations task only starts after both research phases are done.
*   **Robust Tooling**: Define and integrate at least two mock tools (e.g., `SearchTool`, `AnalysisTool`, `ReportWriterTool`) that your agents will use. These tools should simulate real-world capabilities.
*   **Structured Output**: The final output of the crew should be a well-structured market research report, clearly presenting findings and recommendations.

### Evaluation Criteria

Your submission will be evaluated based on:

*   **Correctness**: Proper implementation and usage of `memory`, `evaluation_function`, and `Flow`.
*   **Completeness**: All required steps of the market research (overview, deep dive, recommendations) are addressed.
*   **Quality of Output**: The final market research report is coherent, relevant, and well-structured.
*   **Code Clarity**: The code is well-organized, readable, and includes meaningful comments explaining design choices.
*   **Efficiency**: Agents and tasks are defined logically to minimize redundant work and maximize collaboration.


In [ ]:
import os
from crewai import Agent, Task, Crew, Process, Flow
from crewai_tools import tool
from langchain_openai import ChatOpenAI

# --- Configuration and Mock Setup ---

# Set up your API key for the LLM. In a real scenario, use environment variables.
# For this exercise, we'll use a mock LLM or a placeholder.
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# Mock LLM for demonstration purposes if you don't want to use a real API key
# In a production setting, you would replace this with a configured ChatOpenAI instance.
class MockLLM:
    def invoke(self, prompt):
        print(f"\n--- Mock LLM Invoked ---\nPrompt: {prompt[:200]}...\n")
        if "market overview" in prompt.lower():
            return "Mock Market Overview: Key players include Google Assistant, Apple Siri, Amazon Alexa. Features: voice commands, smart home integration. Pricing: often free, tied to devices."
        elif "competitor deep dive" in prompt.lower():
            return "Mock Competitor Deep Dive: Google Assistant strengths: vast knowledge base, Android integration. Weaknesses: privacy concerns. Siri strengths: Apple ecosystem integration. Weaknesses: limited third-party support."
        elif "recommendations" in prompt.lower():
            return "Mock Recommendations: Focus on niche market, advanced privacy features, unique multimodal interaction."
        else:
            return "Mock LLM Response: This is a generic response for the given prompt."

# Use a real LLM if OPENAI_API_KEY is set, otherwise use the mock LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7) # Recommended for 2026
# llm = MockLLM() # Uncomment to use the mock LLM

# --- Mock Tools Definition ---

@tool("Search Tool")
def search_tool(query: str) -> str:
    """Simulates a web search for market data and competitive intelligence."""
    print(f"\n--- Search Tool Used: {query[:100]}... ---\n")
    if "AI personal assistant market" in query:
        return "Found data on AI personal assistant market: Google Assistant, Apple Siri, Amazon Alexa are dominant. Features include scheduling, information retrieval, smart home control. Pricing is typically device-dependent or free."
    elif "Google Assistant strengths weaknesses" in query:
        return "Google Assistant strengths: extensive knowledge graph, seamless integration with Google services. Weaknesses: data privacy concerns, less personalized than some alternatives."
    elif "Apple Siri strengths weaknesses" in query:
        return "Apple Siri strengths: deep integration with Apple ecosystem, strong privacy focus. Weaknesses: limited third-party app support, sometimes less accurate than competitors."
    elif "Amazon Alexa strengths weaknesses" in query:
        return "Amazon Alexa strengths: vast smart home device compatibility, extensive skill library. Weaknesses: often perceived as less intelligent for complex queries, privacy concerns."
    else:
        return f"No specific search results for '{query}'. Generic search result: AI market is growing, competition is fierce."

@tool("Analysis Tool")
def analysis_tool(data: str) -> str:
    """Simulates data analysis and synthesis from raw information."""
    print(f"\n--- Analysis Tool Used: {data[:100]}... ---\n")
    if "Google Assistant" in data and "Siri" in data:
        return "Synthesized analysis: Google and Apple dominate, each with ecosystem lock-in. Google excels in data, Apple in privacy. Alexa leads in smart home integration."
    elif "market overview" in data:
        return "Initial market analysis: The AI assistant market is mature but still evolving, with opportunities in niche applications or enhanced privacy features."
    else:
        return "Generic analysis: The provided data suggests trends towards personalization and multimodal interaction."

@tool("Report Writer Tool")
def report_writer_tool(content: str) -> str:
    """Simulates writing a structured report based on analyzed content."""
    print(f"\n--- Report Writer Tool Used: {content[:100]}... ---\n")
    return f"Formatted Report Section:\n------------------------\n{content}\n------------------------\n"

# --- Custom Evaluation Functions ---

def evaluate_market_overview(output: str) -> bool:
    """Evaluates if the market overview contains key players and features."""
    required_keywords = ["Google Assistant", "Apple Siri", "Amazon Alexa", "features", "pricing"]
    if all(keyword.lower() in output.lower() for keyword in required_keywords):
        print("Market Overview Evaluation: PASSED - Contains key players and features.")
        return True
    print("Market Overview Evaluation: FAILED - Missing key players or features.")
    return False

def evaluate_competitive_deep_dive(output: str) -> bool:
    """Evaluates if the competitive deep dive covers strengths and weaknesses for at least two competitors."""
    required_phrases = ["strengths:", "weaknesses:"]
    competitors_found = 0
    if "Google Assistant" in output: competitors_found += 1
    if "Apple Siri" in output: competitors_found += 1
    if "Amazon Alexa" in output: competitors_found += 1

    if competitors_found >= 2 and all(phrase.lower() in output.lower() for phrase in required_phrases):
        print("Competitive Deep Dive Evaluation: PASSED - Covers strengths/weaknesses for multiple competitors.")
        return True
    print("Competitive Deep Dive Evaluation: FAILED - Insufficient competitor analysis or missing strengths/weaknesses.")
    return False

def evaluate_strategic_recommendations(output: str) -> bool:
    """Evaluates if recommendations are present and actionable."""
    required_keywords = ["recommendations", "strategy", "positioning"]
    if all(keyword.lower() in output.lower() for keyword in required_keywords) and len(output) > 100:
        print("Strategic Recommendations Evaluation: PASSED - Contains actionable recommendations.")
        return True
    print("Strategic Recommendations Evaluation: FAILED - Missing recommendations or too brief.")
    return False

print("Setup complete. Mock tools and evaluation functions are ready.")


### Your Implementation

Now it's your turn! Implement the `Crew` according to the requirements outlined above. Define your agents, tasks, and orchestrate them using `Flows`. Remember to enable `memory=True` for agents and assign the custom `evaluation_function` to relevant tasks.

Your code should:

1.  Define at least three agents: `Market Analyst`, `Competitive Intelligence Specialist`, and `Report Writer`.
2.  Define tasks for Market Overview, Competitive Deep Dive, and Strategic Recommendations.
3.  Assign the mock tools (`search_tool`, `analysis_tool`, `report_writer_tool`) to the appropriate agents.
4.  Enable `memory=True` for all agents.
5.  Assign the custom `evaluation_function`s (`evaluate_market_overview`, `evaluate_competitive_deep_dive`, `evaluate_strategic_recommendations`) to their respective tasks.
6.  Define a `Flow` to manage the execution order of these tasks.
7.  Instantiate and run the `Crew`.
8.  Print the final output of the crew.


In [ ]:
import os
from crewai import Agent, Task, Crew, Process, Flow
from crewai_tools import tool
from langchain_openai import ChatOpenAI

# --- Configuration and Mock Setup (repeated for self-contained solution) ---

# Use a real LLM if OPENAI_API_KEY is set, otherwise use the mock LLM.
# For this solution, we'll assume OPENAI_API_KEY is set or use a mock for demonstration.
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY" # Uncomment and set your key

class MockLLM:
    def invoke(self, prompt):
        # print(f"\n--- Mock LLM Invoked ---\nPrompt: {prompt[:200]}...\n")
        if "market overview" in prompt.lower():
            return "Mock Market Overview: Key players include Google Assistant, Apple Siri, Amazon Alexa. Features: voice commands, smart home integration. Pricing: often free, tied to devices. Market is mature but growing, with focus on personalization."
        elif "competitor deep dive" in prompt.lower():
            return "Mock Competitor Deep Dive: Google Assistant strengths: vast knowledge base, Android integration. Weaknesses: privacy concerns, generic. Siri strengths: Apple ecosystem integration, strong privacy. Weaknesses: limited third-party support, less accurate. Alexa strengths: smart home, vast skills. Weaknesses: less intelligent for complex queries, privacy."
        elif "recommendations" in prompt.lower():
            return "Mock Recommendations: Based on market analysis, recommend focusing on a niche market (e.g., healthcare professionals), emphasizing advanced privacy features, and developing unique multimodal interaction capabilities (e.g., gesture control). Strategy should highlight data security and specialized knowledge domains. Positioning: 'The Secure & Specialized AI Assistant'."
        else:
            return "Mock LLM Response: This is a generic response for the given prompt."

# Choose your LLM. For production, use ChatOpenAI with your API key.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7) # Recommended for 2026
# llm = MockLLM() # Uncomment to use the mock LLM for testing without API key

# --- Mock Tools Definition (repeated for self-contained solution) ---

@tool("Search Tool")
def search_tool(query: str) -> str:
    """Simulates a web search for market data and competitive intelligence."""
    # print(f"\n--- Search Tool Used: {query[:100]}... ---\n")
    if "AI personal assistant market" in query:
        return "Found data on AI personal assistant market: Google Assistant, Apple Siri, Amazon Alexa are dominant. Features include scheduling, information retrieval, smart home control. Pricing is typically device-dependent or free. Emerging trends: hyper-personalization, multimodal interaction, enhanced privacy."
    elif "Google Assistant strengths weaknesses" in query:
        return "Google Assistant strengths: extensive knowledge graph, seamless integration with Google services. Weaknesses: data privacy concerns, sometimes feels generic."
    elif "Apple Siri strengths weaknesses" in query:
        return "Apple Siri strengths: deep integration with Apple ecosystem, strong privacy focus. Weaknesses: limited third-party app support, often less accurate than competitors for complex queries."
    elif "Amazon Alexa strengths weaknesses" in query:
        return "Amazon Alexa strengths: vast smart home device compatibility, extensive skill library. Weaknesses: often perceived as less intelligent for complex queries, privacy concerns regarding always-on listening."
    else:
        return f"No specific search results for '{query}'. Generic search result: AI market is growing, competition is fierce. New entrants focus on niche markets or specialized features."

@tool("Analysis Tool")
def analysis_tool(data: str) -> str:
    """Simulates data analysis and synthesis from raw information."""
    # print(f"\n--- Analysis Tool Used: {data[:100]}... ---\n")
    if "Google Assistant" in data and "Siri" in data:
        return "Synthesized analysis: Google and Apple dominate, each with ecosystem lock-in. Google excels in data, Apple in privacy. Alexa leads in smart home integration. The market shows a clear demand for more specialized and privacy-respecting AI assistants."
    elif "market overview" in data:
        return "Initial market analysis: The AI assistant market is mature but still evolving, with significant opportunities in niche applications or enhanced privacy features. Key players are well-established, but innovation gaps exist."
    else:
        return "Generic analysis: The provided data suggests trends towards personalization, multimodal interaction, and specialized applications for AI assistants."

@tool("Report Writer Tool")
def report_writer_tool(content: str) -> str:
    """Simulates writing a structured report based on analyzed content."""
    # print(f"\n--- Report Writer Tool Used: {content[:100]}... ---\n")
    return f"\n### Report Section:\n------------------------\n{content}\n------------------------\n"

# --- Custom Evaluation Functions (repeated for self-contained solution) ---

def evaluate_market_overview(output: str) -> bool:
    """Evaluates if the market overview contains key players and features."""
    required_keywords = ["Google Assistant", "Apple Siri", "Amazon Alexa", "features", "pricing"]
    if all(keyword.lower() in output.lower() for keyword in required_keywords):
        print("\n[Evaluation] Market Overview: PASSED - Contains key players and features.")
        return True
    print("\n[Evaluation] Market Overview: FAILED - Missing key players or features.")
    return False

def evaluate_competitive_deep_dive(output: str) -> bool:
    """Evaluates if the competitive deep dive covers strengths and weaknesses for at least two competitors."""
    required_phrases = ["strengths:", "weaknesses:"]
    competitors_found = 0
    if "Google Assistant" in output: competitors_found += 1
    if "Apple Siri" in output: competitors_found += 1
    if "Amazon Alexa" in output: competitors_found += 1

    if competitors_found >= 2 and all(phrase.lower() in output.lower() for phrase in required_phrases):
        print("\n[Evaluation] Competitive Deep Dive: PASSED - Covers strengths/weaknesses for multiple competitors.")
        return True
    print("\n[Evaluation] Competitive Deep Dive: FAILED - Insufficient competitor analysis or missing strengths/weaknesses.")
    return False

def evaluate_strategic_recommendations(output: str) -> bool:
    """Evaluates if recommendations are present and actionable."""
    required_keywords = ["recommendations", "strategy", "positioning"]
    if all(keyword.lower() in output.lower() for keyword in required_keywords) and len(output) > 100:
        print("\n[Evaluation] Strategic Recommendations: PASSED - Contains actionable recommendations.")
        return True
    print("\n[Evaluation] Strategic Recommendations: FAILED - Missing recommendations or too brief.")
    return False

# --- Agent Definitions ---

# Agent 1: Market Analyst
market_analyst = Agent(
    role='Senior Market Analyst',
    goal='Conduct a comprehensive market overview of AI personal assistants, identifying key players, features, and pricing models.',
    backstory='An expert in market research with a keen eye for emerging trends and competitive landscapes. Specializes in synthesizing vast amounts of data into actionable insights.',
    verbose=True,
    allow_delegation=False,
    tools=[search_tool, analysis_tool],
    llm=llm,
    memory=True # Enable memory for context retention
)

# Agent 2: Competitive Intelligence Specialist
competitive_specialist = Agent(
    role='Competitive Intelligence Specialist',
    goal='Perform deep-dive analysis on the top 3 identified competitors, detailing their strengths, weaknesses, and unique selling propositions.',
    backstory='A meticulous researcher focused on competitive benchmarking. Excels at dissecting competitor strategies and identifying areas for differentiation.',
    verbose=True,
    allow_delegation=False,
    tools=[search_tool, analysis_tool],
    llm=llm,
    memory=True # Enable memory for context retention
)

# Agent 3: Report Writer
report_writer = Agent(
    role='Strategic Report Writer',
    goal='Synthesize all market research and competitive analysis into a structured report, including strategic recommendations for the new AI assistant.',
    backstory='A seasoned communicator skilled in translating complex data into clear, concise, and actionable business reports. Focuses on strategic implications and recommendations.',
    verbose=True,
    allow_delegation=False,
    tools=[report_writer_tool],
    llm=llm,
    memory=True # Enable memory for context retention
)

# --- Task Definitions ---

# Task 1: Market Overview
market_overview_task = Task(
    description=(
        "Conduct a thorough market overview of the AI personal assistant sector. "
        "Identify the leading platforms (e.g., Google Assistant, Apple Siri, Amazon Alexa), "
        "their primary features, typical pricing structures, and target demographics. "
        "Summarize current market trends and potential gaps. "
        "The final output should be a detailed summary of the market landscape."
    ),
    expected_output='A comprehensive summary of the AI personal assistant market, including key players, features, pricing, and market trends.',
    agent=market_analyst,
    evaluation_function=evaluate_market_overview, # Assign custom evaluation
    human_input=False
)

# Task 2: Competitive Deep Dive
competitive_deep_dive_task = Task(
    description=(
        "Based on the market overview, select the top 3 most relevant competitors. "
        "For each competitor, perform a deep-dive analysis to identify their core strengths, "
        "significant weaknesses, and unique selling propositions. "
        "Provide a comparative analysis highlighting key differentiators. "
        "The final output should be a detailed competitive analysis for each selected competitor."
    ),
    expected_output='A detailed comparative analysis of the top 3 AI personal assistant competitors, outlining their strengths, weaknesses, and USPs.',
    agent=competitive_specialist,
    context=[market_overview_task], # This task depends on the market overview
    evaluation_function=evaluate_competitive_deep_dive, # Assign custom evaluation
    human_input=False
)

# Task 3: Strategic Recommendations Report
strategic_recommendations_task = Task(
    description=(
        "Synthesize the findings from both the market overview and competitive deep dive. "
        "Generate a strategic market research report that includes: "
        "1. An executive summary of the market landscape. "
        "2. Key insights from the competitive analysis. "
        "3. Actionable recommendations for the new AI assistant's product positioning, "
        "   feature set, and potential market entry strategy. "
        "The report should be professional, concise, and data-driven."
    ),
    expected_output='A professional market research report with an executive summary, competitive insights, and actionable strategic recommendations for the new AI assistant.',
    agent=report_writer,
    context=[market_overview_task, competitive_deep_dive_task], # Depends on both previous tasks
    evaluation_function=evaluate_strategic_recommendations, # Assign custom evaluation
    human_input=False
)

# --- Flow Definition ---

# Define the flow for the market research process
# This ensures tasks run in the correct order with dependencies met.
market_research_flow = Flow(
    name="Market Research Workflow for New AI Assistant",
    description="Orchestrates the market analysis, competitive deep dive, and strategic recommendations for a new AI assistant product.",
    tasks=[
        market_overview_task,
        competitive_deep_dive_task,
        strategic_recommendations_task
    ],
    # CrewAI automatically handles dependencies defined in task.context
    # For more complex conditional logic, you could define explicit steps and conditions.
)

# --- Crew Definition and Execution ---

# Instantiate your crew with a sequential process
# For production, consider Process.hierarchical for more complex coordination.
market_research_crew = Crew(
    agents=[
        market_analyst,
        competitive_specialist,
        report_writer
    ],
    tasks=market_research_flow.tasks, # Use tasks from the defined flow
    process=Process.sequential, # Tasks run in the order defined by the flow/dependencies
    verbose=True,
    manager_llm=llm # Manager LLM is useful for hierarchical process, but good practice to define.
)

print("\n### Starting the Market Research Crew ###")
result = market_research_crew.kickoff()

print("\n### Market Research Complete! ###")
print("\nFinal Report:\n")
print(result)
